In [ ]:
"""
PIPELINE STAGE 4: Structured Overlap Feature Calculation
--------------------------------------------------------
Role in Thesis:
    Calculates analytical overlap metrics based on the formulated sets:
    Precision = |G_i ∩ R_i| / |G_i|
    Recall    = |G_i ∩ R_i| / |R_i|
    
Edge-Case Handling:
    Strictly enforces limits where denominators approach zero (e.g., if |R_i| == 0, Recall = 0.0).
    
Outputs:
    Case-level mismatch summaries used as observable indicators for the Bayes model.
"""

In [ ]:
# =====================================================================
# CALCULATION OF STRUCTURED OVERLAP METRICS
# =====================================================================
# Operationalizes the set-theoretic comparison equations from the thesis:
# Precision = |G_i ∩ R_i| / |G_i|
# Recall    = |G_i ∩ R_i| / |R_i|
# F1-Score  = Harmonic mean of Precision and Recall

# Edge Case Handling (Division by Zero):
# Mathematical limits are enforced here via 'replace(0, np.nan)' or safe division.
# If |R_i| == 0, Recall is explicitly handled to prevent undefined behavior.

# =====================================================================
# STRUCTURING HALLUCINATION CANDIDATES (G_i \ R_i)
# =====================================================================
# Isolates observations present ONLY in the generated output (generated_only_count).
# These act as raw clinical candidates for unsupported findings (hallucinations),
# which will later form the basis for manual expert review.

# =========================
# 1. Imports
# =========================
import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


# =========================
# 2. Config
# =========================
NOTEBOOK_DIR = Path.cwd().resolve()

def find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "outputs").exists() and (p / "data").exists():
            return p
    raise FileNotFoundError("Could not find project root.")

BASE_DIR = find_project_root(NOTEBOOK_DIR)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("BASE_DIR:", BASE_DIR)

RADGRAPH_DIR = BASE_DIR / "outputs" / "radgraph_extractions_1500_subset"
ALIGNED_INPUT_JSONL = BASE_DIR / "outputs" / "radgraph_input_1500_subset.jsonl"

GEN_CSV = RADGRAPH_DIR / "generated_reports_radgraph_processed.csv"
REF_CSV = RADGRAPH_DIR / "reference_reports_radgraph_processed.csv"

OUT_DIR = BASE_DIR / "outputs" / "radgraph_analysis_1500_subset"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CASE_COMPARISON_CSV = OUT_DIR / "case_level_comparison.csv"
OBS_FREQUENCY_CSV = OUT_DIR / "observation_frequency_summary.csv"
OBS_ERROR_CSV = OUT_DIR / "observation_error_summary.csv"
QUALITATIVE_CASES_CSV = OUT_DIR / "qualitative_cases.csv"
SUMMARY_TXT = OUT_DIR / "analysis_summary.txt"


# =========================
# 3. Helper functions
# =========================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\n", " ").replace("\r", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_obs(x):
    x = normalize_text(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x

def unique_sorted_nonempty(values):
    vals = []
    for v in values:
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.append(s)
    return sorted(set(vals))

def series_to_obs_set(series):
    vals = [normalize_obs(v) for v in series.dropna().tolist()]
    vals = [v for v in vals if v]
    return set(vals)

def safe_join(items):
    items = [str(x).strip() for x in items if str(x).strip()]
    return "; ".join(items)

def top_n(counter_obj, n=15):
    return counter_obj.most_common(n)


# =========================
# 4. Load files
# =========================
assert GEN_CSV.exists(), f"Missing file: {GEN_CSV}"
assert REF_CSV.exists(), f"Missing file: {REF_CSV}"
assert ALIGNED_INPUT_JSONL.exists(), f"Missing file: {ALIGNED_INPUT_JSONL}"

gen_df = pd.read_csv(GEN_CSV)
ref_df = pd.read_csv(REF_CSV)
aligned_df = pd.read_json(ALIGNED_INPUT_JSONL, lines=True)

print("gen_df shape:", gen_df.shape)
print("ref_df shape:", ref_df.shape)
print("aligned_df shape:", aligned_df.shape)

display(gen_df.head(3))
display(ref_df.head(3))
display(aligned_df.head(3))


# =========================
# 5. Clean key columns
# =========================
for df_ in [gen_df, ref_df, aligned_df]:
    df_["case_id"] = df_["case_id"].astype(str).str.strip()

if "generated_report" in aligned_df.columns:
    aligned_df["generated_report"] = aligned_df["generated_report"].map(normalize_text)
if "reference_report" in aligned_df.columns:
    aligned_df["reference_report"] = aligned_df["reference_report"].map(normalize_text)

if "observation" in gen_df.columns:
    gen_df["observation_norm"] = gen_df["observation"].map(normalize_obs)
if "observation" in ref_df.columns:
    ref_df["observation_norm"] = ref_df["observation"].map(normalize_obs)


# =========================
# 6. Case-level comparison
# =========================
case_ids = sorted(set(aligned_df["case_id"].tolist()))

rows = []

for case_id in case_ids:
    gen_case = gen_df.loc[gen_df["case_id"] == case_id].copy()
    ref_case = ref_df.loc[ref_df["case_id"] == case_id].copy()
    align_case = aligned_df.loc[aligned_df["case_id"] == case_id].copy()

    gen_obs = set(gen_case["observation_norm"].dropna()) if "observation_norm" in gen_case.columns else set()
    ref_obs = set(ref_case["observation_norm"].dropna()) if "observation_norm" in ref_case.columns else set()

    gen_obs = {x for x in gen_obs if x}
    ref_obs = {x for x in ref_obs if x}

    shared = sorted(gen_obs & ref_obs)
    generated_only = sorted(gen_obs - ref_obs)
    reference_only = sorted(ref_obs - gen_obs)

    generated_report = align_case["generated_report"].iloc[0] if len(align_case) else ""
    reference_report = align_case["reference_report"].iloc[0] if len(align_case) else ""

    precision = len(shared) / len(gen_obs) if len(gen_obs) else np.nan
    recall = len(shared) / len(ref_obs) if len(ref_obs) else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    jaccard = len(shared) / len(gen_obs | ref_obs) if len(gen_obs | ref_obs) else np.nan

    rows.append({
        "case_id": case_id,
        "generated_report": generated_report,
        "reference_report": reference_report,
        "generated_observation_count": len(gen_obs),
        "reference_observation_count": len(ref_obs),
        "shared_observation_count": len(shared),
        "generated_only_count": len(generated_only),
        "reference_only_count": len(reference_only),
        "precision_obs": precision,
        "recall_obs": recall,
        "f1_obs": f1,
        "jaccard_obs": jaccard,
        "shared_observations": safe_join(shared),
        "generated_only": safe_join(generated_only),
        "reference_only": safe_join(reference_only),
    })

case_comp_df = pd.DataFrame(rows).sort_values(
    by=["f1_obs", "shared_observation_count", "reference_only_count"],
    ascending=[False, False, True]
).reset_index(drop=True)

case_comp_df.to_csv(CASE_COMPARISON_CSV, index=False)

print("saved:", CASE_COMPARISON_CSV)
display(case_comp_df.head(10))


# =========================
# 7. Observation frequencies
# =========================
gen_counter = Counter([x for x in gen_df["observation_norm"].dropna().tolist() if x])
ref_counter = Counter([x for x in ref_df["observation_norm"].dropna().tolist() if x])

all_obs = sorted(set(gen_counter.keys()) | set(ref_counter.keys()))

freq_rows = []
for obs in all_obs:
    gen_count = gen_counter.get(obs, 0)
    ref_count = ref_counter.get(obs, 0)
    freq_rows.append({
        "observation": obs,
        "generated_count": gen_count,
        "reference_count": ref_count,
        "count_diff_generated_minus_reference": gen_count - ref_count,
    })

obs_freq_df = pd.DataFrame(freq_rows).sort_values(
    by=["reference_count", "generated_count", "observation"],
    ascending=[False, False, True]
).reset_index(drop=True)

obs_freq_df.to_csv(OBS_FREQUENCY_CSV, index=False)

print("saved:", OBS_FREQUENCY_CSV)
display(obs_freq_df.head(20))


# =========================
# 8. Observation error summary
# =========================
extra_counter = Counter()
missing_counter = Counter()
shared_counter = Counter()

for _, row in case_comp_df.iterrows():
    shared = [x.strip() for x in str(row["shared_observations"]).split(";") if x.strip()]
    extra = [x.strip() for x in str(row["generated_only"]).split(";") if x.strip()]
    missing = [x.strip() for x in str(row["reference_only"]).split(";") if x.strip()]

    shared_counter.update(shared)
    extra_counter.update(extra)
    missing_counter.update(missing)

error_rows = []
all_error_obs = sorted(set(shared_counter.keys()) | set(extra_counter.keys()) | set(missing_counter.keys()))

for obs in all_error_obs:
    error_rows.append({
        "observation": obs,
        "shared_case_count": shared_counter.get(obs, 0),
        "generated_only_case_count": extra_counter.get(obs, 0),
        "reference_only_case_count": missing_counter.get(obs, 0),
    })

obs_error_df = pd.DataFrame(error_rows).sort_values(
    by=["reference_only_case_count", "generated_only_case_count", "shared_case_count", "observation"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

obs_error_df.to_csv(OBS_ERROR_CSV, index=False)

print("saved:", OBS_ERROR_CSV)
display(obs_error_df.head(20))


# =========================
# 9. Qualitative sample selection
# =========================
best_cases = case_comp_df.sort_values(
    by=["f1_obs", "shared_observation_count"],
    ascending=[False, False]
).head(5).copy()
best_cases["case_category"] = "best_match"

worst_cases = case_comp_df.sort_values(
    by=["f1_obs", "reference_only_count", "generated_only_count"],
    ascending=[True, False, False]
).head(5).copy()
worst_cases["case_category"] = "worst_match"

hallucination_cases = case_comp_df.sort_values(
    by=["generated_only_count", "f1_obs"],
    ascending=[False, True]
).head(5).copy()
hallucination_cases["case_category"] = "many_generated_only"

omission_cases = case_comp_df.sort_values(
    by=["reference_only_count", "f1_obs"],
    ascending=[False, True]
).head(5).copy()
omission_cases["case_category"] = "many_reference_only"

qual_df = pd.concat(
    [best_cases, worst_cases, hallucination_cases, omission_cases],
    ignore_index=True
).drop_duplicates(subset=["case_id", "case_category"])

qual_df = qual_df[
    [
        "case_category",
        "case_id",
        "generated_observation_count",
        "reference_observation_count",
        "shared_observation_count",
        "generated_only_count",
        "reference_only_count",
        "precision_obs",
        "recall_obs",
        "f1_obs",
        "jaccard_obs",
        "shared_observations",
        "generated_only",
        "reference_only",
        "generated_report",
        "reference_report",
    ]
].reset_index(drop=True)

qual_df.to_csv(QUALITATIVE_CASES_CSV, index=False)

print("saved:", QUALITATIVE_CASES_CSV)
display(qual_df.head(20))


# =========================
# 10. High-level metrics
# =========================
overall = {
    "num_cases": int(len(case_comp_df)),
    "avg_generated_observation_count": float(case_comp_df["generated_observation_count"].mean()),
    "avg_reference_observation_count": float(case_comp_df["reference_observation_count"].mean()),
    "avg_shared_observation_count": float(case_comp_df["shared_observation_count"].mean()),
    "avg_generated_only_count": float(case_comp_df["generated_only_count"].mean()),
    "avg_reference_only_count": float(case_comp_df["reference_only_count"].mean()),
    "mean_precision_obs": float(case_comp_df["precision_obs"].mean(skipna=True)),
    "mean_recall_obs": float(case_comp_df["recall_obs"].mean(skipna=True)),
    "mean_f1_obs": float(case_comp_df["f1_obs"].mean(skipna=True)),
    "mean_jaccard_obs": float(case_comp_df["jaccard_obs"].mean(skipna=True)),
}

overall_df = pd.DataFrame([overall])
display(overall_df)

top_missing = pd.DataFrame(top_n(missing_counter, 15), columns=["observation", "reference_only_case_count"])
top_extra = pd.DataFrame(top_n(extra_counter, 15), columns=["observation", "generated_only_case_count"])
top_shared = pd.DataFrame(top_n(shared_counter, 15), columns=["observation", "shared_case_count"])

print("Top missing observations")
display(top_missing)

print("Top extra observations")
display(top_extra)

print("Top shared observations")
display(top_shared)


# =========================
# 11. Write summary text
# =========================
summary_lines = []
summary_lines.append("RadGraph analysis summary")
summary_lines.append("=" * 80)
summary_lines.append(f"Cases analyzed: {overall['num_cases']}")
summary_lines.append(f"Mean generated observation count: {overall['avg_generated_observation_count']:.3f}")
summary_lines.append(f"Mean reference observation count: {overall['avg_reference_observation_count']:.3f}")
summary_lines.append(f"Mean shared observation count: {overall['avg_shared_observation_count']:.3f}")
summary_lines.append(f"Mean generated-only observation count: {overall['avg_generated_only_count']:.3f}")
summary_lines.append(f"Mean reference-only observation count: {overall['avg_reference_only_count']:.3f}")
summary_lines.append(f"Mean observation precision: {overall['mean_precision_obs']:.3f}")
summary_lines.append(f"Mean observation recall: {overall['mean_recall_obs']:.3f}")
summary_lines.append(f"Mean observation F1: {overall['mean_f1_obs']:.3f}")
summary_lines.append(f"Mean observation Jaccard: {overall['mean_jaccard_obs']:.3f}")
summary_lines.append("")
summary_lines.append("Top missing observations:")
for _, r in top_missing.iterrows():
    summary_lines.append(f"- {r['observation']}: {int(r['reference_only_case_count'])}")
summary_lines.append("")
summary_lines.append("Top extra observations:")
for _, r in top_extra.iterrows():
    summary_lines.append(f"- {r['observation']}: {int(r['generated_only_case_count'])}")
summary_lines.append("")
summary_lines.append("Top shared observations:")
for _, r in top_shared.iterrows():
    summary_lines.append(f"- {r['observation']}: {int(r['shared_case_count'])}")

SUMMARY_TXT.write_text("\n".join(summary_lines), encoding="utf-8")

print("saved:", SUMMARY_TXT)


# =========================
# 12. Preview thesis-ready subsets
# =========================
print("Best 5 cases by observation F1")
display(case_comp_df.sort_values(by=["f1_obs", "shared_observation_count"], ascending=[False, False]).head(5))

print("Worst 5 cases by observation F1")
display(case_comp_df.sort_values(by=["f1_obs", "reference_only_count", "generated_only_count"], ascending=[True, False, False]).head(5))

print("Most frequent missing observations")
display(obs_error_df.sort_values(by=["reference_only_case_count", "generated_only_case_count"], ascending=[False, False]).head(10))

print("Most frequent extra observations")
display(obs_error_df.sort_values(by=["generated_only_case_count", "reference_only_case_count"], ascending=[False, False]).head(10))


NOTEBOOK_DIR: /workspace/thesis/notebooks
BASE_DIR: /workspace/thesis
gen_df shape: (2097, 6)
ref_df shape: (793, 6)
aligned_df shape: (272, 9)


,case_id,source_type,observation,located_at,suggestive_of,tags
0,1004616657379357,generated,wire fractured,NaN,NaN,definitely present
1,1004616657379357,generated,normal,heart size,NaN,definitely present
2,1004616657379357,generated,unremarkable,mediastinal and hilar contours,NaN,definitely present


,case_id,source_type,observation,located_at,suggestive_of,tags
0,1004616657379357,reference,findings,NaN,findings suggestive of pneumonia,definitely absent
1,1004616657379357,reference,pneumonia,NaN,NaN,definitely absent
2,1026887750239281,reference,left picc,NaN,NaN,definitely present


,case_id,study_id,dicom_id,image_path,generated_report,reference_report,hypothesis,reference,match_source
0,1004616657379357,NaN,NaN,mimic/p10/p10046166/s57379357/6e511483-c7e1601...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,id
1,1026887750239281,NaN,NaN,mimic/p10/p10268877/s50239281/0c69d156-6f5f3a8...,Single AP upright portable view of the chest w...,1. Left PICC tip appears to terminate in the d...,Single AP upright portable view of the chest w...,1. Left PICC tip appears to terminate in the d...,id
2,1026887751513702,NaN,NaN,mimic/p10/p10268877/s51513702/053e0fdd-17dbee8...,Single AP upright portable view of the chest w...,No definite acute cardiopulmonary process. Enl...,Single AP upright portable view of the chest w...,No definite acute cardiopulmonary process. Enl...,id


saved: /workspace/thesis/outputs/radgraph_analysis_1500_subset/case_level_comparison.csv


...

,case_id,generated_report,reference_report,generated_observation_count,reference_observation_count,shared_observation_count,generated_only_count,reference_only_count,precision_obs,recall_obs,f1_obs,jaccard_obs,shared_observations,generated_only,reference_only
66,1105293553884408,PA and lateral views of the chest. The cardiac...,Slight improvement in ill-defined patchy opaci...,10,7,1,9,6,0.100000,0.142857,0.117647,0.062500,unchanged,acute abnormalities; changes; emphysematous; l...,atelectasis; ill - defined opacity; infection;...
65,1211086353008088,Single portable view of the chest. There is ne...,Mild pulmonary vascular congestion with probab...,11,6,1,10,5,0.090909,0.166667,0.117647,0.062500,atelectasis,blunting; clips; confluent opacity; degenerati...,airspace; chronic changes; mild congestion; sm...
64,1093360956535476,The heart is normal in size. The mediastinal a...,Bilateral upper lobe scarring with upward retr...,9,7,1,8,6,0.111111,0.142857,0.125000,0.066667,patchy opacity,edema; effusion; mild process; normal; pneumon...,atelectasis; infection; prior tuberculosis; sa...
63,1102224550078440,"Endotracheal tube is seen, terminating approxi...",1. Bilateral airspace opacity consistent with ...,12,4,1,11,3,0.083333,0.250000,0.125000,0.066667,opacity,coiling; endotracheal tube; enlarged; hemorrha...,moderate cardiomegaly; nodular opacity; pneumonia
62,1147406559155076,A single portable radiograph of the chest was ...,1. Persistent but improved mild pulmonary edem...,11,5,1,10,4,0.090909,0.200000,0.125000,0.066667,effusion,calcified lymph nodes; changes; congestion; mi...,improved mild edema; loculated; loculated effu...


Most frequent missing observations


,observation,shared_case_count,generated_only_case_count,reference_only_case_count
0,acute process,0,0,41
1,pneumonia,9,20,37
2,atelectasis,14,37,28
3,infection,3,8,17
4,edema,4,32,14
5,consolidation,0,16,13
6,opacity,2,7,11
7,cardiomegaly,0,2,11
8,mild,0,1,11
9,scarring,1,8,10


Most frequent extra observations


,observation,shared_case_count,generated_only_case_count,reference_only_case_count
362,pneumothorax,6,175,0
13,effusion,7,131,8
74,clear,0,58,1
363,normal,0,56,0
75,focal consolidation,1,52,1
76,unchanged,1,48,1
364,unremarkable,0,45,0
2,atelectasis,14,37,28
365,large effusion,1,34,0
4,edema,4,32,14
